# 8.数据规整:连接、联合和重塑

# 8.1 层次化索引

In [1]:
import pandas as pd
import numpy as np
data = pd.Series(np.random.uniform(size=9),index=[['a','a','a','b','b','c','c','d','d'],[1,2,3,1,3,1,2,2,3]])
data

a  1    0.864297
   2    0.320274
   3    0.552777
b  1    0.234145
   3    0.687210
c  1    0.111248
   2    0.142407
d  2    0.037723
   3    0.338341
dtype: float64

In [2]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [3]:
data['b']

1    0.234145
3    0.687210
dtype: float64

In [4]:
data['b':'c']

b  1    0.234145
   3    0.687210
c  1    0.111248
   2    0.142407
dtype: float64

In [5]:
data.loc[['b','d']]

b  1    0.234145
   3    0.687210
d  2    0.037723
   3    0.338341
dtype: float64

In [6]:
data.loc[:,2]

a    0.320274
c    0.142407
d    0.037723
dtype: float64

In [7]:
# 重排到DataFrame
data.unstack()

,1,2,3
a,0.864297,0.320274,0.552777
b,0.234145,NaN,0.687210
c,0.111248,0.142407,NaN
d,NaN,0.037723,0.338341


In [8]:
data.unstack().stack()

a  1    0.864297
   2    0.320274
   3    0.552777
b  1    0.234145
   2         NaN
   3    0.687210
c  1    0.111248
   2    0.142407
   3         NaN
d  1         NaN
   2    0.037723
   3    0.338341
dtype: float64

In [10]:
frame = pd.DataFrame(np.arange(12).reshape(4,3),index=[['a','a','b','b'],[1,2,1,2]],columns=[['Ohio','Ohio','Colorado'],['Green','Red','Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [19]:
frame.index.names = ['key1','key2']
frame.columns.names = ['state','color']
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [13]:
frame.index.nlevels

2

In [14]:
frame['Ohio']

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

In [15]:
# 单独创建然后复用
pd.MultiIndex.from_arrays([['Ohio','Ohio','Colorado'],['Green','Red','Green']],names=['state','color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

## 8.1.1 重排序和层级排序

In [16]:
frame.swaplevel('key1','key2')

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [17]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [18]:
frame.swaplevel(0,1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

## 8.1.2 按层级进行汇总统计

In [20]:
frame.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

## 8.1.3 使用DataFrame的列进行索引

In [23]:
frame = pd.DataFrame({'a':range(7),'b':range(7,0,-1),'c':['one','one','one','two','two','two','two'],'d':[0,1,2,0,1,2,3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [24]:
frame2 = frame.set_index(['c','d'])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [25]:
# 默认情况下,这些列会从DataFrame移除,但也可以通过传入drop=False将其保留下来
frame.set_index(['c','d'],drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

In [26]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


# End